# clip-grad-norm-pre-step — faded example 2: Compute the global L2 norm in a manual clip

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `clip-grad-norm-pre-step`. The last cell reports your progress on the `Optimizer: clip_grad_norm pre-step` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: clip_grad_norm pre-step` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`clip-grad-norm-pre-step`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "clip-grad-norm-pre-step"
DD_SUBTOPIC = "Optimizer: clip_grad_norm pre-step"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The global L2 norm clipped by `clip_grad_norm_` is `sqrt(sum over all gradients of (g**2).sum())` - one scalar across the entire parameter group, not per-tensor. Rescaling by `max_norm / total` (when `total > max_norm`) caps the combined update while keeping direction.

## Faded exercise 2

### Faded - compute the global norm

This is a from-scratch reimplementation of `clip_grad_norm_`. The collection of grads, the threshold check, the in-place rescale, and the return are all written. Fill in the single line that computes the global L2 norm `total` from the list of detached grads.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
t.manual_seed(0)

def manual_clip(params, max_norm):
    grads = [p.grad for p in params if p.grad is not None]
    if not grads:
        return 0.0
    total = None  # TODO: fill in this step — read the prompt cell above
    if total.item() > max_norm:
        scale = max_norm / (total + 1e-6)
        for g in grads:
            g.mul_(scale)
    return total.item()

a = t.zeros(3, requires_grad=True)
b = t.zeros(2, requires_grad=True)
a.grad = t.tensor([2.0, 2.0, 2.0])
b.grad = t.tensor([2.0, 2.0])
print("pre-clip norm:", round(manual_clip([a, b], max_norm=3.0), 4))
print("post a.grad norm contribution", round(a.grad.norm().item(), 4))


def _test():
    import torch.nn.utils as nn_utils
    # ground truth via torch reference on identical grads
    a = t.zeros(3, requires_grad=True)
    b = t.zeros(2, requires_grad=True)
    a.grad = t.tensor([2.0, 2.0, 2.0])
    b.grad = t.tensor([2.0, 2.0])
    got = manual_clip([a, b], max_norm=3.0)
    # pre-clip norm: sqrt(3*4 + 2*4) = sqrt(20)
    expected_pre = (20.0) ** 0.5
    assert abs(got - expected_pre) < 1e-4, f"expected pre-clip {expected_pre}, got {got}"
    # torch reference for the post-clip grads
    ar = t.zeros(3, requires_grad=True)
    br = t.zeros(2, requires_grad=True)
    ar.grad = t.tensor([2.0, 2.0, 2.0])
    br.grad = t.tensor([2.0, 2.0])
    nn_utils.clip_grad_norm_([ar, br], max_norm=3.0)
    assert t.allclose(a.grad, ar.grad, atol=1e-4), "post-clip grad a mismatch vs torch"
    assert t.allclose(b.grad, br.grad, atol=1e-4), "post-clip grad b mismatch vs torch"


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
t.manual_seed(0)

def manual_clip(params, max_norm):
    grads = [p.grad for p in params if p.grad is not None]
    if not grads:
        return 0.0
    total = sum((g.detach() ** 2).sum() for g in grads).sqrt()
    if total.item() > max_norm:
        scale = max_norm / (total + 1e-6)
        for g in grads:
            g.mul_(scale)
    return total.item()

a = t.zeros(3, requires_grad=True)
b = t.zeros(2, requires_grad=True)
a.grad = t.tensor([2.0, 2.0, 2.0])
b.grad = t.tensor([2.0, 2.0])
print("pre-clip norm:", round(manual_clip([a, b], max_norm=3.0), 4))
print("post a.grad norm contribution", round(a.grad.norm().item(), 4))
```
</details>